# Evaluate Wild-Deepfake Corruption Levels 1-5 With Best-4 TTA Methods

Notebook này chạy 4 method TTA đã chọn trên các Kaggle datasets Wild-Deepfake corruption level 1-5:

- `dota_bw0.55_m0.95_ct0.9`
- `free_bw0.6_m0.9_p1`
- `bca_t0.03_bw0.7_pm0.95_xm0.98`
- `tda_pa0.4_pb5.5_pe0.4_na0`

Vì dataset chưa có embedding, notebook sẽ:

1. Quét ảnh theo cấu trúc `corruption/level_x/wild-deepfake/{real,fake}`.
2. Extract CLIP global embeddings và cache thành `.pt`.
3. Load FF++ train embeddings để fit TTA cache/prototypes.
4. Eval 4 TTA methods với linear probe model.
5. Lưu summary metrics và optional per-sample probabilities.


## Kaggle Setup


In [ ]:
# Chạy trên Kaggle nếu repo chưa có trong /kaggle/working.
# !git clone -b dev https://github.com/hoavien0110/training-free-tta-for-deepfake-detection.git /kaggle/working/training-free-tta-for-deepfake-detection
# %cd /kaggle/working/training-free-tta-for-deepfake-detection
# !pip install -q -e . --no-deps
# !pip install -q open_clip_torch


## Imports


In [ ]:
from pathlib import Path
import math
import sys
from types import SimpleNamespace

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    confusion_matrix,
    f1_score,
    roc_auc_score,
    roc_curve,
)
from tqdm.auto import tqdm


def find_repo_root():
    candidates = [Path.cwd(), *Path.cwd().parents, Path('/kaggle/working/training-free-tta-for-deepfake-detection')]
    for root in candidates:
        if (root / 'code' / 'deepfake_tta').exists():
            return root.resolve(), (root / 'code').resolve()
        if (root / 'deepfake_tta').exists():
            return root.resolve(), root.resolve()
    raise FileNotFoundError('Cannot find repo root containing deepfake_tta')

repo_root, code_root = find_repo_root()
sys.path.insert(0, str(code_root))

from deepfake_tta.modeling import load_feature_file, predict_probe_scores, seed_everything
from testing.evaluate_tta_matrix import load_probe_model, read_thresholds
from testing.evaluate_tta_param_sweep import create_method
from training.ffpp_split_utils import create_clip, extract_features, save_feature_file

print('repo_root:', repo_root)
print('code_root:', code_root)


## Config


In [ ]:
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
SEED = 42

ON_KAGGLE = Path('/kaggle/input').exists()
if ON_KAGGLE:
    FFPP_SPLIT = Path('/kaggle/input/ffpp-split-features')
    MODELS_DIR = Path('/kaggle/input/ffpp-training-free-models')
    OUTPUT_DIR = Path('/kaggle/working/wild_deepfake_level1_5_best4_tta')
else:
    FFPP_SPLIT = repo_root / 'embeddings' / 'ffpp-split-features'
    MODELS_DIR = repo_root / 'models'
    OUTPUT_DIR = repo_root / 'result' / 'wild_deepfake_level1_5_best4_tta'

TRAIN_FEATURES = FFPP_SPLIT / 'ffpp_train_features.pt'
MODEL_SPECS = {
    'linear-probe': MODELS_DIR / 'ffpp_linear_probe_split.pt',
    # Bật nếu muốn chạy thêm OSD cùng 4 method.
    # 'osd': MODELS_DIR / 'ffpp_osd_linear_probe_split.pt',
}
THRESHOLDS_CSV = [MODELS_DIR / 'ffpp_probe_thresholds.csv', MODELS_DIR / 'thresholds.csv']

LEVELS = [1, 2, 3, 4, 5]
CORRUPTIONS = ['color_contrast', 'color_saturation', 'gaussian_blur', 'resize']
DATASET_FOLDER_NAME = 'wild-deepfake'

CLIP_MODEL = 'ViT-L-14'
PRETRAINED = 'openai'
EXTRACT_BATCH_SIZE = 128
EXTRACT_NUM_WORKERS = 2
USE_AMP = True
SKIP_EXISTING_FEATURES = True
MAX_SAMPLES_PER_TARGET = None  # set 2000 để smoke test nhanh.
SHUFFLE_TARGET = False

EVAL_BATCH_SIZE = 4096
TEST_BATCH_SIZE = 512
CACHE_BATCH_SIZE = 8192
BALANCE_METHOD_FIT = True
SAVE_PROBES = True
CONTINUE_ON_ERROR = True

FEATURE_DIR = OUTPUT_DIR / 'features'
SUMMARY_OUTPUT = OUTPUT_DIR / 'wild_deepfake_level1_5_best4_tta_summary.csv'
PROBES_OUTPUT = OUTPUT_DIR / 'wild_deepfake_level1_5_best4_tta_samples.csv'

METHOD_CONFIGS = [
    {'method': 'dota', 'param_id': 'dota_bw0.55_m0.95_ct0.9', 'base_weight': 0.55, 'momentum': 0.95, 'confidence_threshold': 0.9},
    {'method': 'freetta', 'param_id': 'free_bw0.6_m0.9_p1', 'base_weight': 0.6, 'momentum': 0.9, 'prior_power': 1.0},
    {'method': 'bca', 'param_id': 'bca_t0.03_bw0.7_pm0.95_xm0.98', 'temperature': 0.03, 'base_weight': 0.7, 'prior_momentum': 0.95, 'prototype_momentum': 0.98},
    {'method': 'tda', 'param_id': 'tda_pa0.4_pb5.5_pe0.4_na0', 'positive_alpha': 0.4, 'positive_beta': 5.5, 'positive_entropy_threshold': 0.4, 'negative_alpha': 0.0},
]

TTA_ARGS = SimpleNamespace(
    seed=SEED,
    eval_batch_size=EVAL_BATCH_SIZE,
    test_batch_size=TEST_BATCH_SIZE,
    cache_batch_size=CACHE_BATCH_SIZE,
    freetta_min_var=1e-4,
    freetta_warmup_batches=1,
    bca_confidence_threshold=0.0,
    dota_min_var=1e-4,
    tda_positive_shot_capacity=64,
    tda_negative_beta=5.5,
    tda_negative_shot_capacity=64,
    tda_negative_entropy_lower=0.35,
    tda_negative_entropy_upper=0.8,
    tda_negative_mask_lower=0.2,
    tda_negative_mask_upper=0.8,
    tda_top_k=64,
    gda_min_var=1e-4,
    etta_entropy_power=1.0,
    etta_base_weight=1.0,
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FEATURE_DIR.mkdir(parents=True, exist_ok=True)
seed_everything(SEED)

print('device:', DEVICE)
print('train features:', TRAIN_FEATURES)
print('models:', MODEL_SPECS)
print('output:', OUTPUT_DIR)


## Resolve Wild Dataset Folders


In [ ]:
IMAGE_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}


def has_wild_structure(root, level):
    root = Path(root)
    for corruption in CORRUPTIONS:
        base = root / corruption / f'level_{level}' / DATASET_FOLDER_NAME
        if (base / 'real').exists() or (base / 'fake').exists():
            return True
    return False


def find_wild_root(level):
    if not ON_KAGGLE:
        candidates = [repo_root / 'data' / f'wild-deepfake-level-{level}']
    else:
        candidates = [
            Path(f'/kaggle/input/wild-deepfake-corruption-level-{level}/wild-deepfake-level-{level}'),
            Path(f'/kaggle/input/wild-deepfake-corruption-level-{level}'),
            Path(f'/kaggle/input/wild-deepfake-level-{level}'),
        ]
        # Fallback for Kaggle mount names that differ from the slug.
        candidates.extend(Path('/kaggle/input').glob(f'*/wild-deepfake-level-{level}'))
        candidates.extend(Path('/kaggle/input').glob(f'*level-{level}'))
    seen = set()
    for candidate in candidates:
        candidate = Path(candidate)
        if candidate in seen:
            continue
        seen.add(candidate)
        if candidate.exists() and has_wild_structure(candidate, level):
            return candidate
    raise FileNotFoundError(
        f'Cannot find Wild-Deepfake level {level}. Expected e.g. '
        f'/kaggle/input/wild-deepfake-corruption-level-{level}/wild-deepfake-level-{level}/color_contrast/level_{level}/wild-deepfake/{{real,fake}}'
    )


def image_files(folder):
    folder = Path(folder)
    if not folder.exists():
        return []
    return sorted(p for p in folder.rglob('*') if p.is_file() and p.suffix.lower() in IMAGE_EXTS)


def build_wild_dataframe(root, level, corruption):
    base = Path(root) / corruption / f'level_{level}' / DATASET_FOLDER_NAME
    rows = []
    for label_name, label_num in [('real', 0), ('fake', 1)]:
        for path in image_files(base / label_name):
            rows.append({
                'imagepath_fixed': str(path),
                'label_num': label_num,
                'label': label_name.upper(),
                'dataset': f'wild-deepfake-level{level}-{corruption}',
                'level': level,
                'corruption': corruption,
            })
    df = pd.DataFrame(rows)
    if df.empty:
        raise FileNotFoundError(f'No images found under {base}')
    if MAX_SAMPLES_PER_TARGET is not None and len(df) > MAX_SAMPLES_PER_TARGET:
        parts = []
        per_class = MAX_SAMPLES_PER_TARGET // 2
        for label_num in [0, 1]:
            sub = df[df['label_num'].eq(label_num)]
            parts.append(sub.sample(n=min(per_class, len(sub)), random_state=SEED))
        df = pd.concat(parts, ignore_index=True)
    if SHUFFLE_TARGET:
        df = df.sample(frac=1, random_state=SEED).reset_index(drop=True)
    return df.reset_index(drop=True)

wild_roots = {level: find_wild_root(level) for level in LEVELS}
print('Wild roots:')
for level, root in wild_roots.items():
    print(' level', level, '->', root)

target_specs = []
for level in LEVELS:
    for corruption in CORRUPTIONS:
        target_specs.append({'level': level, 'corruption': corruption, 'root': wild_roots[level]})

preview_rows = []
for spec in target_specs:
    df = build_wild_dataframe(spec['root'], spec['level'], spec['corruption'])
    counts = df['label_num'].value_counts().sort_index().to_dict()
    preview_rows.append({**spec, 'rows': len(df), 'real': counts.get(0, 0), 'fake': counts.get(1, 0)})
preview = pd.DataFrame(preview_rows)
display(preview)


## Extract Or Load Target Embeddings


In [ ]:
def feature_output_path(level, corruption):
    return FEATURE_DIR / f'wild_deepfake_level{level}_{corruption}_features.pt'

clip_model = None
preprocess = None
feature_paths = []

for spec in target_specs:
    level = spec['level']
    corruption = spec['corruption']
    out_path = feature_output_path(level, corruption)
    feature_paths.append(out_path)
    if SKIP_EXISTING_FEATURES and out_path.exists():
        print('skip existing feature:', out_path)
        continue

    if clip_model is None:
        clip_model, preprocess = create_clip(CLIP_MODEL, PRETRAINED, DEVICE)
        if DEVICE == 'cuda':
            torch.backends.cudnn.benchmark = True

    df = build_wild_dataframe(spec['root'], level, corruption)
    print()
    print('extract:', f'level={level}', corruption, '| rows:', len(df), '| counts:', df['label_num'].value_counts().sort_index().to_dict())
    features, labels, paths = extract_features(
        df,
        clip_model=clip_model,
        preprocess=preprocess,
        device=DEVICE,
        batch_size=EXTRACT_BATCH_SIZE,
        num_workers=EXTRACT_NUM_WORKERS,
        use_amp=USE_AMP,
    )
    save_feature_file(
        out_path,
        features,
        labels,
        paths,
        metadata={
            'dataset_name': 'Wild-Deepfake',
            'split_name': 'test',
            'clip_model': f'{CLIP_MODEL}/{PRETRAINED}',
            'transform_name': corruption,
            'transform_level': level,
            'source_root': str(spec['root']),
        },
    )

print('feature files:')
for path in feature_paths:
    print(' -', path, 'exists=', path.exists())


## Load Source Train Features And Model


In [ ]:
def select_balanced_subset(feats, labels, *, seed, name):
    labels = labels.long()
    counts = torch.bincount(labels, minlength=2)
    keep = int(counts.min().item())
    if keep <= 0:
        raise ValueError(f'Cannot balance {name}: {counts.tolist()}')
    g = torch.Generator().manual_seed(seed)
    selected = []
    for label in [0, 1]:
        idx = torch.where(labels.eq(label))[0]
        selected.append(idx[torch.randperm(len(idx), generator=g)[:keep]])
    idx = torch.cat(selected)
    idx = idx[torch.randperm(len(idx), generator=g)]
    print(f'balanced {name}:', counts.tolist(), '->', torch.bincount(labels[idx], minlength=2).tolist())
    return feats[idx].contiguous(), labels[idx].contiguous()


def threshold_for_model(thresholds, model_name, model_type, model_path):
    model_path = Path(model_path)
    return thresholds.get(
        model_name,
        thresholds.get(
            f'{model_name}:{model_type}',
            thresholds.get(f'{model_name}:{model_path.name}', thresholds.get(model_path.name, thresholds.get(model_type, 0.5))),
        ),
    )

assert TRAIN_FEATURES.exists(), f'Missing TRAIN_FEATURES: {TRAIN_FEATURES}'
for name, path in MODEL_SPECS.items():
    assert Path(path).exists(), f'Missing model {name}: {path}'

train_feats, train_labels, train_payload = load_feature_file(str(TRAIN_FEATURES))
if BALANCE_METHOD_FIT:
    train_feats, train_labels = select_balanced_subset(train_feats, train_labels, seed=SEED + 101, name='method fit')

thresholds = read_thresholds([str(p) for p in THRESHOLDS_CSV if Path(p).exists()])
print('thresholds:', thresholds)

loaded_models = {}
for model_name, model_path in MODEL_SPECS.items():
    model_type, model = load_probe_model(Path(model_path), train_feats.shape[1], DEVICE)
    loaded_models[model_name] = (model_type, model, Path(model_path), threshold_for_model(thresholds, model_name, model_type, model_path))
    print('loaded model:', model_name, model_type, model_path, 'threshold:', loaded_models[model_name][3])


## Evaluation Helpers


In [ ]:
def calculate_eer(y_true, y_score):
    if len(np.unique(y_true)) < 2:
        return np.nan, np.nan
    fpr, tpr, thresholds = roc_curve(y_true, y_score)
    fnr = 1.0 - tpr
    idx = int(np.nanargmin(np.abs(fnr - fpr)))
    return float((fpr[idx] + fnr[idx]) / 2.0), float(thresholds[idx])


def summarize_scores(y_true, y_score, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    eer, eer_threshold = calculate_eer(y_true, y_score)
    return {
        'acc': float(accuracy_score(y_true, y_pred)),
        'f1': float(f1_score(y_true, y_pred, average='macro', zero_division=0)),
        'auc': float(roc_auc_score(y_true, y_score)) if len(np.unique(y_true)) == 2 else np.nan,
        'ap': float(average_precision_score(y_true, y_score)) if len(np.unique(y_true)) == 2 else np.nan,
        'eer': eer,
        'eer_threshold': eer_threshold,
        'tn': int(tn),
        'fp': int(fp),
        'fn': int(fn),
        'tp': int(tp),
    }


def sample_ids_from_payload(payload, n):
    paths = payload.get('paths')
    if paths is None:
        return [f'idx_{idx:06d}' for idx in range(n)]
    return [str(path) for path in paths]


def append_csv(path, rows):
    if not rows:
        return
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    current = pd.DataFrame(rows)
    if path.exists():
        old = pd.read_csv(path)
        current = pd.concat([old, current], ignore_index=True, sort=False)
    current.to_csv(path, index=False)


def probe_rows(common, sample_ids, labels, probs, y_pred, threshold, fit_samples, fit_counts):
    y_true = labels.detach().cpu().numpy().astype(int)
    probs_np = probs.detach().cpu().numpy()
    rows = []
    for idx, sample_id in enumerate(sample_ids):
        prob_real = float(probs_np[idx, 0])
        prob_fake = float(probs_np[idx, 1])
        rows.append({
            **common,
            'sample_index': idx,
            'sample_id': sample_id,
            'y_true': int(y_true[idx]),
            'prob_real': prob_real,
            'prob_fake': prob_fake,
            'score': prob_fake,
            'y_pred': int(y_pred[idx]),
            'correct': bool(int(y_pred[idx]) == int(y_true[idx])),
            'confidence': float(max(prob_real, prob_fake)),
            'margin': float(abs(prob_fake - prob_real)),
            'threshold': float(threshold),
            'method_fit_samples': fit_samples,
            'method_fit_counts': fit_counts,
        })
    return rows


## Run 4 TTA Methods


In [ ]:
summary_rows = []
if SAVE_PROBES and PROBES_OUTPUT.exists():
    PROBES_OUTPUT.unlink()
if SUMMARY_OUTPUT.exists():
    SUMMARY_OUTPUT.unlink()

for feature_path in feature_paths:
    feats, labels, payload = load_feature_file(str(feature_path))
    y_true = labels.detach().cpu().numpy().astype(int)
    sample_ids = sample_ids_from_payload(payload, len(labels))
    level = payload.get('transform_level')
    corruption = payload.get('transform_name', 'none')
    dataset_name = f'wild-deepfake-level{level}-{corruption}'
    print()
    print('===', dataset_name, '===')

    for model_name, (model_type, model, model_path, threshold) in loaded_models.items():
        for config in METHOD_CONFIGS:
            common = {
                'dataset': dataset_name,
                'payload_dataset': payload.get('dataset_name', 'Wild-Deepfake'),
                'split': payload.get('split_name', 'test'),
                'level': level,
                'corruption': corruption,
                'feature_path': str(feature_path),
                'model': model_name,
                'model_type': model_type,
                'model_path': str(model_path),
                **config,
            }
            try:
                print('eval:', dataset_name, model_name, config['param_id'], flush=True)
                method, fit_samples, fit_counts = create_method(config, TTA_ARGS, train_feats, train_labels)
                probs = method.predict_proba(model, feats, DEVICE).detach().cpu().float()
                scores = probs[:, 1].numpy()
                y_pred = probs.argmax(dim=1).numpy().astype(int)
                metrics = summarize_scores(y_true, scores, y_pred)
                summary_rows.append({
                    **common,
                    'n_samples': int(len(labels)),
                    'label_counts': torch.bincount(labels.long(), minlength=2).tolist(),
                    'method_fit_samples': fit_samples,
                    'method_fit_counts': fit_counts,
                    **metrics,
                })
                if SAVE_PROBES:
                    append_csv(PROBES_OUTPUT, probe_rows(common, sample_ids, labels, probs, y_pred, threshold, fit_samples, fit_counts))
            except Exception as exc:
                row = {**common, 'n_samples': int(len(labels)), 'error': repr(exc)}
                summary_rows.append(row)
                print('ERROR:', repr(exc), flush=True)
                if not CONTINUE_ON_ERROR:
                    raise
            pd.DataFrame(summary_rows).to_csv(SUMMARY_OUTPUT, index=False)

summary = pd.DataFrame(summary_rows)
summary.to_csv(SUMMARY_OUTPUT, index=False)
print('saved summary:', SUMMARY_OUTPUT)
if SAVE_PROBES:
    print('saved probes:', PROBES_OUTPUT)
display(summary.sort_values(['level', 'corruption', 'model', 'method']))


## Quick Summary


In [ ]:
valid = summary[summary.get('error').isna()] if 'error' in summary.columns else summary
metric_cols = ['acc', 'f1', 'auc', 'ap', 'eer']
if not valid.empty:
    display(valid.groupby(['method', 'param_id'])[metric_cols].mean().sort_values('auc', ascending=False))
    display(valid.sort_values(['level', 'corruption', 'auc'], ascending=[True, True, False]))
